# Amazon Reviews — Sentiment & Visual Analysis

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'font.family': 'sans-serif'})

In [ ]:
df = pd.read_csv('../Datasets/Amazon_Product_Review.csv', encoding='latin1')

# fill missing ratings with median
df['reviews.rating'] = df['reviews.rating'].fillna(df['reviews.rating'].median())
df['reviews.numHelpful'] = df['reviews.numHelpful'].fillna(0)
df['reviews.username'].fillna('Anonymous', inplace=True)

# sentiment label
def label(r):
    if r >= 4:   return 'Positive'
    elif r == 3: return 'Neutral'
    return 'Negative'

df['sentiment'] = df['reviews.rating'].apply(label)
print(f'Rows: {len(df)}  |  Sentiment counts:')
print(df['sentiment'].value_counts().to_string())

---
## 1 — Histogram: Rating Distribution

In [ ]:
colors = {1: '#d32f2f', 2: '#f57c00', 3: '#fbc02d', 4: '#388e3c', 5: '#1565c0'}
rating_counts = df['reviews.rating'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    [str(int(r)) + ' ★' for r in rating_counts.index],
    rating_counts.values,
    color=[colors[int(r)] for r in rating_counts.index],
    width=0.55,
    edgecolor='white',
    linewidth=1.2
)

# value labels on top of each bar
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 8, f'{int(h):,}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Rating Distribution', fontsize=15, fontweight='bold', pad=14)
ax.set_xlabel('Star Rating', fontsize=12)
ax.set_ylabel('Number of Reviews', fontsize=12)
ax.set_ylim(0, rating_counts.max() * 1.15)
ax.spines[['top', 'right']].set_visible(False)

# legend
patches = [mpatches.Patch(color=colors[i], label=f'{i} Star') for i in sorted(colors)]
ax.legend(handles=patches, loc='upper left', framealpha=0.6, fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/charts/hist_rating_distribution.png', dpi=150)
plt.show()

---
## 2 — Pie / Donut: Sentiment Breakdown

In [ ]:
sent_counts = df['sentiment'].value_counts()
sent_colors = {'Positive': '#43a047', 'Neutral': '#fdd835', 'Negative': '#e53935'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Donut ---
wedges, texts, autotexts = axes[0].pie(
    sent_counts,
    labels=sent_counts.index,
    colors=[sent_colors[s] for s in sent_counts.index],
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.78,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2)
)
for t in autotexts:
    t.set_fontsize(11)
    t.set_fontweight('bold')
axes[0].set_title('Sentiment Share (Donut)', fontsize=13, fontweight='bold')

# --- Horizontal bar ---
order = ['Positive', 'Neutral', 'Negative']
vals  = [sent_counts.get(s, 0) for s in order]
bars  = axes[1].barh(order, vals, color=[sent_colors[s] for s in order],
                     height=0.45, edgecolor='white')
for bar in bars:
    w = bar.get_width()
    axes[1].text(w + 5, bar.get_y() + bar.get_height() / 2,
                 f'{int(w):,}', va='center', fontsize=11, fontweight='bold')
axes[1].set_title('Sentiment Count', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Reviews', fontsize=11)
axes[1].set_xlim(0, max(vals) * 1.12)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('Sentiment Categorization', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/charts/sentiment_breakdown.png', dpi=150)
plt.show()

---
## 3 — Bar Plot: Top 15 Most Reviewed Products

In [ ]:
top15 = (
    df.groupby('name')
      .agg(review_count=('reviews.text', 'count'),
           avg_rating=('reviews.rating', 'mean'))
      .sort_values('review_count', ascending=False)
      .head(15)
      .reset_index()
)

# colour each bar by avg rating tier
def rating_color(r):
    if r >= 4.5:  return '#1565c0'
    elif r >= 4:  return '#388e3c'
    elif r >= 3:  return '#fdd835'
    return '#e53935'

bar_colors = [rating_color(r) for r in top15['avg_rating']]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(
    top15['name'][::-1],
    top15['review_count'][::-1],
    color=bar_colors[::-1],
    height=0.6,
    edgecolor='white'
)

for bar, rating in zip(bars, top15['avg_rating'][::-1]):
    w = bar.get_width()
    ax.text(w + 1, bar.get_y() + bar.get_height() / 2,
            f'{int(w)} reviews  |  ★ {rating:.1f}',
            va='center', fontsize=8.5)

ax.set_title('Top 15 Most Reviewed Products', fontsize=15, fontweight='bold', pad=14)
ax.set_xlabel('Number of Reviews', fontsize=12)
ax.set_xlim(0, top15['review_count'].max() * 1.3)
ax.spines[['top', 'right']].set_visible(False)

legend_patches = [
    mpatches.Patch(color='#1565c0', label='★ 4.5+'),
    mpatches.Patch(color='#388e3c', label='★ 4.0–4.4'),
    mpatches.Patch(color='#fdd835', label='★ 3.0–3.9'),
    mpatches.Patch(color='#e53935', label='★ < 3.0'),
]
ax.legend(handles=legend_patches, title='Avg Rating', loc='lower right',
          framealpha=0.7, fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/charts/bar_top_products.png', dpi=150)
plt.show()

---
## 4 — Heatmap: Review Correlations

In [ ]:
# encode sentiment as numeric for correlation
df['sentiment_score'] = df['sentiment'].map({'Positive': 1, 'Neutral': 0, 'Negative': -1})

corr_cols = {
    'reviews.rating'  : 'Rating',
    'reviews.numHelpful': 'Helpful Votes',
    'sentiment_score' : 'Sentiment',
}
corr_df = df[list(corr_cols.keys())].rename(columns=corr_cols)
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True   # hide upper triangle duplicates

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    linewidths=1.5,
    linecolor='white',
    square=True,
    annot_kws={'size': 13, 'weight': 'bold'},
    ax=ax
)

ax.set_title('Review Correlation Heatmap', fontsize=14, fontweight='bold', pad=14)
ax.tick_params(axis='x', labelsize=11, rotation=15)
ax.tick_params(axis='y', labelsize=11, rotation=0)
plt.tight_layout()
plt.savefig('../outputs/charts/heatmap_correlations.png', dpi=150)
plt.show()

print('\nCorrelation matrix:')
print(corr_matrix.round(3).to_string())

---
## 5 — Stacked Bar: Sentiment per Product (Top 10)

In [ ]:
top10_names = top15['name'].head(10).tolist()
stacked = (
    df[df['name'].isin(top10_names)]
    .groupby(['name', 'sentiment'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['Positive', 'Neutral', 'Negative'], fill_value=0)
)
stacked = stacked.loc[top10_names]  # keep top10 order

# shorten long product names for readability
stacked.index = [n[:35] + '…' if len(n) > 35 else n for n in stacked.index]

fig, ax = plt.subplots(figsize=(12, 6))
stacked.plot(
    kind='barh',
    stacked=True,
    color=['#43a047', '#fdd835', '#e53935'],
    edgecolor='white',
    ax=ax
)

ax.set_title('Sentiment Breakdown — Top 10 Products', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Number of Reviews', fontsize=12)
ax.set_ylabel('')
ax.legend(title='Sentiment', loc='lower right', fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/charts/stacked_sentiment_top10.png', dpi=150)
plt.show()